# Label-aware DEC deduplication walkthrough

This notebook removes redundant samples using identical decompiled representations. It reads the labels supplied with RanDS, computes a digest for every DEC artifact, resolves label disagreements inside each digest group, and materializes the retained EXE, DIS, and DEC token-ID files.

## 1. Deduplication flow

```text
Benign.csv / Ransomware.csv ── SHA-256 join ──────────────┐
                                                           │
notebook 01 DEC artifacts ── content MD5 ── digest groups ┼─ label-aware selection
                                                           │          │
notebook 02 EXE/DIS/DEC token IDs ─────────────────────────┘          │
                                                                      ├─ benign + ransomware: drop whole group
                                                                      └─ one binary class: keep one sample with modal family/label
                                                                                  │
                                                                                  └─ filtered EXE/DIS/DEC token files
```

DEC is the deduplication basis for every model input. The same retained SHA-256 set is applied to EXE, DIS, and DEC, so the three representations remain aligned.

## 2. Environment and imported pipeline functions

The data package owns digest calculation, RanDS metadata parsing, cross-class conflict removal, modal-label selection, and token-file materialization. The notebook supplies the experiment paths and displays intermediate results.

The vocabulary size must match the tokenized files produced by notebook 02.

In [ ]:
from collections import Counter, defaultdict, OrderedDict
import json
from pathlib import Path
from pprint import pprint
import re
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from malweave.data import (
    deduplicate_labeled_files,
    get_digests_from_files,
    load_rands_metadata,
    materialize_token_files,
)

PREPROCESSED_ROOT = PROJECT_ROOT / 'data' / 'ranDS' / 'processed' / 'reproduction'
TOKENIZED_ROOT = PROJECT_ROOT / 'data' / 'ranDS' / 'processed' / 'tokenized'
DEDUPLICATED_ROOT = PROJECT_ROOT / 'data' / 'ranDS' / 'processed' / 'deduplicated'
BENIGN_CSV = PROJECT_ROOT.parent / 'Benign.csv'
RANSOMWARE_CSV = PROJECT_ROOT.parent / 'Ransomware.csv'
VOCAB_SIZE = 16_384
SHA256_PATTERN = re.compile(r'^[0-9a-f]{64}$')

assert BENIGN_CSV.is_file(), f'Missing metadata: {BENIGN_CSV}'
assert RANSOMWARE_CSV.is_file(), f'Missing metadata: {RANSOMWARE_CSV}'
assert PREPROCESSED_ROOT.is_dir(), 'Run notebook 01 first.'
assert TOKENIZED_ROOT.is_dir(), 'Run notebook 02 first.'

print('Project root:', PROJECT_ROOT)
print('DEC digest input:', PREPROCESSED_ROOT / 'dec')
print('Token-ID input:', TOKENIZED_ROOT)
print('Deduplicated output:', DEDUPLICATED_ROOT)
print('Vocabulary size:', f'{VOCAB_SIZE:,}')

Project root: /Users/jung/Projects/malrec-lab/malweave
DEC digest input: /Users/jung/Projects/malrec-lab/malweave/data/ranDS/processed/reproduction/dec
Token-ID input: /Users/jung/Projects/malrec-lab/malweave/data/ranDS/processed/tokenized
Deduplicated output: /Users/jung/Projects/malrec-lab/malweave/data/ranDS/processed/deduplicated
Vocabulary size: 16,384


## 3. Load binary and ransomware-family labels

The source metadata file supplies the binary label: `Benign.csv` means `benign`, and `Ransomware.csv` means `ransomware`. Ransomware rows also supply the family used to resolve disagreements among samples with identical DEC.

The loader handles the observed RanDS header shift where the values under `Family`, `Packed`, and `Entropy` are actually packed status, entropy, and family respectively.

In [ ]:
sample_metadata, metadata_counts = load_rands_metadata(BENIGN_CSV, RANSOMWARE_CSV)
print('Metadata counts:', metadata_counts)
print('Total unique SHA-256 values:', f'{len(sample_metadata):,}')

manifest_records = [
    json.loads(line)
    for line in (PREPROCESSED_ROOT / 'manifest.jsonl').read_text().splitlines()
]
kept_samples = sorted(record['name'] for record in manifest_records if record['status'] == 'kept')
missing_labels = [sample for sample in kept_samples if sample not in sample_metadata]
assert not missing_labels, f'Preprocessed samples without labels: {missing_labels[:5]}'

print(f'Joined {len(kept_samples)} kept preprocessing samples to metadata.')
print('Joined label head:')
for sample in kept_samples[:8]:
    metadata = sample_metadata[sample]
    print(
        f'  {sample[:16]} | binary={metadata.binary_label:<10} '
        f'family={str(metadata.family):<16} year={metadata.year}'
    )

Metadata counts: {'benign': 127141, 'ransomware': 110546}
Total unique SHA-256 values: 237,687
Joined 8 kept preprocessing samples to metadata.
Joined label head:
  d00630d78796caf7 | binary=ransomware family=PornoAsset       year=2012
  d07092d99a764b58 | binary=ransomware family=Loki             year=2023
  d079e9fcd6bb012d | binary=ransomware family=STOP             year=2023
  d098ebd6d83c498e | binary=ransomware family=LockBit          year=2022
  d09ef86191b744e8 | binary=ransomware family=Quasar           year=2021
  d0d2525c3cdd04ca | binary=ransomware family=Razy             year=2018
  d0d8504d8c8baf19 | binary=ransomware family=Loki             year=2023
  d0d87cb5e049db6d | binary=ransomware family=Zeppelin         year=2022


## 4. Inspect DEC inputs and tokenized representations

The DEC files determine duplicate groups. EXE, DIS, and DEC token directories are checked now because every retained SHA-256 must be materialized for all three model inputs. The preview shows real decompiled text and real token IDs without printing complete artifacts.

In [ ]:
dec_files = sorted((PREPROCESSED_ROOT / 'dec').glob('*.c'))
assert dec_files, 'No DEC files found. Run notebook 01 with Ghidra enabled.'
dec_samples = [path.stem for path in dec_files]
assert set(dec_samples) == set(kept_samples), 'DEC files and kept preprocessing records differ.'

TOKEN_DIRECTORIES = OrderedDict({
    representation: TOKENIZED_ROOT / representation / 'bpe' / str(VOCAB_SIZE)
    for representation in ('exe', 'dis', 'dec')
})
for representation, directory in TOKEN_DIRECTORIES.items():
    token_samples = {
        path.stem for path in directory.glob('*.json') if SHA256_PATTERN.fullmatch(path.stem)
    }
    missing = sorted(set(dec_samples) - token_samples)
    assert not missing, f'{representation.upper()} token IDs missing: {missing[:5]}'
    print(f'{representation.upper()} | {len(token_samples)} token files in {directory}')

first_dec = dec_files[0]
print(f'\nDEC example: {first_dec.name} ({first_dec.stat().st_size:,} bytes)')
print('\n'.join(first_dec.read_text(errors='replace').splitlines()[:8]))
for representation, directory in TOKEN_DIRECTORIES.items():
    token_ids = json.loads((directory / f'{first_dec.stem}.json').read_text())
    print(f'{representation.upper()} token-ID head ({len(token_ids):,} tokens):', token_ids[:20])

EXE | 8 token files in /Users/jung/Projects/malrec-lab/malweave/data/ranDS/processed/tokenized/exe/bpe/16384
DIS | 8 token files in /Users/jung/Projects/malrec-lab/malweave/data/ranDS/processed/tokenized/dis/bpe/16384
DEC | 8 token files in /Users/jung/Projects/malrec-lab/malweave/data/ranDS/processed/tokenized/dec/bpe/16384

DEC example: d00630d78796caf768661e92c3f00a404067b033f4f7dce336801ea721ad3a91.c (39,109 bytes)

/* WARNING: Globals starting with '_' overlap smaller symbols at the same address */

void __fastcall FUN_00401000(undefined4 param_1,uint param_2)

{
  undefined *puVar1;
  DWORD DVar2;
EXE token-ID head (3,662 tokens): [3123, 27, 7786, 83, 836, 168, 227, 866, 1659, 131, 835, 91, 836, 13360, 227, 2425, 15307, 226, 2425, 11719]
DIS token-ID head (3,857 tokens): [205, 209, 1733, 7604, 1357, 627, 309, 6731, 1460, 1457, 7047, 85, 102, 245, 2586, 85, 102, 481, 2826, 85]
DEC token-ID head (6,568 tokens): [554, 4779, 4785, 4776, 4781, 4786, 4783, 2384, 2102, 801, 24, 434, 369

## 5. Compute DEC content digests

A digest is a fixed-size fingerprint of the complete DEC bytes. Equal DEC files receive the same MD5 and therefore enter the same species/group. MD5 is used here as an exact-deduplication index, not as a security guarantee.

The original PE SHA-256 remains the sample identifier; the MD5 below describes the processed DEC representation.

In [ ]:
sha_digest_map = get_digests_from_files(dec_files)
assert set(sha_digest_map) == set(dec_samples)

digest_groups = defaultdict(list)
for path in dec_files:
    sample = path.stem
    digest_groups[sha_digest_map[sample]].append(sample)
    metadata = sample_metadata[sample]
    print(
        f'{sample[:12]} | DEC={path.stat().st_size:>9,} bytes '
        f'| md5={sha_digest_map[sample]} | binary={metadata.binary_label} '
        f'| family={metadata.family}'
    )

repeated_species = {
    digest: samples for digest, samples in digest_groups.items() if len(samples) > 1
}
print('\nDEC redundancy summary:')
pprint({
    'samples': len(dec_samples),
    'total_species': len(digest_groups),
    'repeated_species': len(repeated_species),
    'redundant_samples': sum(len(samples) - 1 for samples in repeated_species.values()),
})
if not repeated_species:
    print('No identical DEC groups in the current subset.')
for digest, samples in repeated_species.items():
    print(f'GROUP | {digest} | {len(samples)} samples')
    for sample in samples:
        metadata = sample_metadata[sample]
        print(f'  {sample} | binary={metadata.binary_label} family={metadata.family}')

d00630d78796 | DEC=   39,109 bytes | md5=3a929cb6f8359b1695007055ff002c54 | binary=ransomware | family=PornoAsset
d07092d99a76 | DEC=  481,831 bytes | md5=9a5d20d57f75e6f18d5c4f1c31ae9900 | binary=ransomware | family=Loki
d079e9fcd6bb | DEC=  416,483 bytes | md5=b83eb22d76913fa5762831a601638205 | binary=ransomware | family=STOP
d098ebd6d83c | DEC=  370,933 bytes | md5=10a4d3e2c8983098e9374c03d48250df | binary=ransomware | family=LockBit
d09ef86191b7 | DEC=  192,468 bytes | md5=d142dee2c3249376374c3b0a53d4d458 | binary=ransomware | family=Quasar
d0d2525c3cdd | DEC=1,799,502 bytes | md5=49d5a865ee7337662c317d664a5ff77c | binary=ransomware | family=Razy
d0d8504d8c8b | DEC=9,220,120 bytes | md5=ea4e75c2bc35111c9d1e042b2866ba3a | binary=ransomware | family=Loki
d0d87cb5e049 | DEC=  415,767 bytes | md5=f0abc43d8d79507c38f6f8f00c2dc31a | binary=ransomware | family=Zeppelin

DEC redundancy summary:
{'redundant_samples': 0,
 'repeated_species': 0,
 'samples': 8,
 'total_species': 8}
No identica

## 6. Apply label-aware selection

Selection uses two label levels:

1. If one DEC digest occurs in both binary classes, the complete group is removed. Identical model input cannot safely carry both `benign` and `ransomware`.
2. Otherwise, the most frequent task label in the group is selected. For ransomware this is `Family`; for benign samples it is `benign`. One sample with that modal label represents the digest and the remaining samples are removed.

Input files are sorted by SHA-256. Consequently, when label frequencies tie, the first SHA-256 encountered wins consistently.

In [ ]:
retained_dec_files, decisions, modal_label_by_digest = deduplicate_labeled_files(
    dec_files,
    sample_metadata,
    sha_digest_map,
)
retained_samples = [path.stem for path in retained_dec_files]
action_counts = Counter(decision.action for decision in decisions)
print('Decision counts:', dict(action_counts))
print('Retained samples:', len(retained_samples))

for decision in decisions:
    print(
        f'{decision.action:<21} {decision.sample[:12]} '
        f'| binary={decision.binary_label:<10} task={decision.task_label:<16} '
        f'| modal={str(decision.modal_label):<16} '
        f'| representative={str(decision.representative)[:12]}'
    )

retained_digests = [sha_digest_map[sample] for sample in retained_samples]
assert len(retained_digests) == len(set(retained_digests)), 'A DEC duplicate survived.'

Decision counts: {'keep': 8}
Retained samples: 8
keep                  d00630d78796 | binary=ransomware task=PornoAsset       | modal=PornoAsset       | representative=d00630d78796
keep                  d07092d99a76 | binary=ransomware task=Loki             | modal=Loki             | representative=d07092d99a76
keep                  d079e9fcd6bb | binary=ransomware task=STOP             | modal=STOP             | representative=d079e9fcd6bb
keep                  d098ebd6d83c | binary=ransomware task=LockBit          | modal=LockBit          | representative=d098ebd6d83c
keep                  d09ef86191b7 | binary=ransomware task=Quasar           | modal=Quasar           | representative=d09ef86191b7
keep                  d0d2525c3cdd | binary=ransomware task=Razy             | modal=Razy             | representative=d0d2525c3cdd
keep                  d0d8504d8c8b | binary=ransomware task=Loki             | modal=Loki             | representative=d0d8504d8c8b
keep                  d0d87

## 7. Materialize retained EXE, DIS, and DEC token IDs

The DEC decision produces one global SHA-256 allow-list. Applying that same list to all three token directories preserves sample alignment. Existing sample JSON files in the exact output directory are synchronized with the current decision so stale results cannot survive a rerun.

Each representation receives the retained token arrays plus `digests.json`, a complete decision `manifest.jsonl`, a retained-only `selected.jsonl`, and `summary.json`.

In [ ]:
materialization_summary = materialize_token_files(
    retained_samples=retained_samples,
    decisions=decisions,
    sha_digest_map=sha_digest_map,
    token_directories=TOKEN_DIRECTORIES,
    output_root=DEDUPLICATED_ROOT,
    vocab_size=VOCAB_SIZE,
)
print('Materialization summary:')
pprint(materialization_summary)

Materialization summary:
{'dec': {'deduplication_basis': 'dec',
         'dropped_binary_conflicts': 0,
         'dropped_duplicates': 0,
         'dropped_non_modal_labels': 0,
         'input_samples': 8,
         'kept_samples': 8,
         'representation': 'dec',
         'stale_outputs_removed': 0,
         'vocab_size': 16384},
 'dis': {'deduplication_basis': 'dec',
         'dropped_binary_conflicts': 0,
         'dropped_duplicates': 0,
         'dropped_non_modal_labels': 0,
         'input_samples': 8,
         'kept_samples': 8,
         'representation': 'dis',
         'stale_outputs_removed': 0,
         'vocab_size': 16384},
 'exe': {'deduplication_basis': 'dec',
         'dropped_binary_conflicts': 0,
         'dropped_duplicates': 0,
         'dropped_non_modal_labels': 0,
         'input_samples': 8,
         'kept_samples': 8,
         'representation': 'exe',
         'stale_outputs_removed': 0,
         'vocab_size': 16384}}


## 8. Inspect materialized output

This check reads the files written to disk rather than reusing in-memory token arrays. It verifies sample counts and prints a token-ID head, the retained label record, and the first keep/drop decisions.

In [ ]:
for representation in TOKEN_DIRECTORIES:
    output_directory = DEDUPLICATED_ROOT / representation / 'bpe' / str(VOCAB_SIZE)
    sample_paths = sorted(
        path for path in output_directory.glob('*.json') if SHA256_PATTERN.fullmatch(path.stem)
    )
    assert [path.stem for path in sample_paths] == retained_samples
    first_ids = json.loads(sample_paths[0].read_text())
    selected_head = (output_directory / 'selected.jsonl').read_text().splitlines()[:2]
    manifest_head = (output_directory / 'manifest.jsonl').read_text().splitlines()[:3]
    print(f'\n{representation.upper()} OUTPUT | {len(sample_paths)} samples')
    print('  files:', [path.name[:16] for path in sample_paths[:5]])
    print('  token-ID head:', first_ids[:24])
    print('  selected head:')
    print('\n'.join(f'    {line}' for line in selected_head))
    print('  decision manifest head:')
    print('\n'.join(f'    {line}' for line in manifest_head))


EXE OUTPUT | 8 samples
  files: ['d00630d78796caf7', 'd07092d99a764b58', 'd079e9fcd6bb012d', 'd098ebd6d83c498e', 'd09ef86191b744e8']
  token-ID head: [3123, 27, 7786, 83, 836, 168, 227, 866, 1659, 131, 835, 91, 836, 13360, 227, 2425, 15307, 226, 2425, 11719, 264, 1054, 24, 4478]
  selected head:
    {"binary_label": "ransomware", "dec_digest": "3a929cb6f8359b1695007055ff002c54", "output": "/Users/jung/Projects/malrec-lab/malweave/data/ranDS/processed/deduplicated/exe/bpe/16384/d00630d78796caf768661e92c3f00a404067b033f4f7dce336801ea721ad3a91.json", "sample": "d00630d78796caf768661e92c3f00a404067b033f4f7dce336801ea721ad3a91", "task_label": "PornoAsset", "tokens": 3662, "year": 2012}
    {"binary_label": "ransomware", "dec_digest": "9a5d20d57f75e6f18d5c4f1c31ae9900", "output": "/Users/jung/Projects/malrec-lab/malweave/data/ranDS/processed/deduplicated/exe/bpe/16384/d07092d99a764b583259254d2be9c346c652e747ee58a2e27756a37c42032c48.json", "sample": "d07092d99a764b583259254d2be9c346c652e747e

## 9. Output contract and next step

```text
data/ranDS/processed/deduplicated/
├── exe/bpe/16384/
├── dis/bpe/16384/
└── dec/bpe/16384/
    ├── <sha256>.json   # retained token-ID sequence
    ├── digests.json    # sample -> DEC content digest
    ├── manifest.jsonl  # every keep/drop decision
    ├── selected.jsonl  # retained samples with binary/task labels
    └── summary.json    # before/after counts
```

All three directories contain the same retained SHA-256 set because DEC is the shared deduplication basis. The next dataset stage should create leakage-safe train, validation, and test partitions. In a full experiment, perform deduplication and splitting before fitting the tokenizer so duplicate content cannot influence vocabulary learning across partitions.